### Knowledge Agents

Creates Agent Bricks Knowledge Assistants for every document corpus used by the
operational dashboard supervisor:

- **Inspection KA** — food safety inspection reports
- **Menu KA** — restaurant menu PDFs (items, nutrition, allergens)
- **Legal KA** — employment, liability, vendor disputes
- **Regulatory KA** — permits, fire safety, zoning, FDA
- **Audit KA** — financial, operational, food safety, supply chain audits
- **Consultancy KA** — strategy, operations, AI transformation, workforce

KAs are created in parallel; readiness polling is delegated to the downstream
`Readiness_Check` task.

In [ ]:
%pip install --upgrade databricks-sdk mlflow-skinny[databricks]

In [ ]:
dbutils.library.restartPython()

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")
import sys
sys.path.append('../utils')
from ops_profile import (
    parse_csv_keys,
    KA_KEYS,
    DEFAULT_KA_KEYS,
    ka_display_name,
)

def _job_param(name: str, default: str = "") -> str:
    try:
        return dbutils.widgets.get(name)
    except Exception:
        return default

KA_KEY_LIST = parse_csv_keys(
    _job_param("KA_KEYS", DEFAULT_KA_KEYS),
    KA_KEYS,
    DEFAULT_KA_KEYS,
    "KA_KEYS",
)
print(f"KA keys: {KA_KEY_LIST}")


##### Knowledge Assistant configuration

One config dict per KA. Each is created via the Knowledge Assistants REST API
(`/api/2.1/knowledge-assistants`) in two steps — create the KA, then attach a
`files` knowledge source pointing at the UC Volume — and registered in
`uc_state` for cleanup. Endpoint names are auto-generated by the API and read
back from the create response.

In [ ]:
from databricks.sdk import WorkspaceClient
import json, sys

sys.path.append('../utils')
from uc_state import add

w = WorkspaceClient()

# Knowledge Assistants v2.1 — endpoint names are auto-generated by Databricks
# and read back from the create response (no longer user-supplied).
API_BASE = "/api/2.1/knowledge-assistants"

# Each entry produces one KA + one "files" knowledge source attached to a UC Volume.
# `display_name` must be unique at workspace level (used to look up via List API).
# `source_display_name` shows up in the Databricks UI but does not affect routing.
KA_CONFIGS_ALL = [
    {
        "display_name": f"{CATALOG}-inspection-knowledge",
        "key": "inspection",
        "volume_path": f"/Volumes/{CATALOG}/food_safety/reports",
        "source_display_name": "Inspection Reports",
        "description": (
            "Answers questions about food safety inspections at Casper's Kitchens "
            "ghost kitchen locations. Covers inspection scores, violations, "
            "corrective actions, and compliance status across all 8 locations (US + EMEA)."
        ),
        "source_description": (
            "Food safety inspection report PDFs for 8 ghost kitchen locations (US + EMEA). "
            "Each PDF contains the full inspection report with facility information, overall "
            "score, letter grade, violation details, corrective actions, and follow-up status."
        ),
        "instructions": (
            "You are a food safety compliance assistant for Casper's Kitchens. "
            "Always cite the specific inspection report (location and date) when answering. "
            "Be precise about violation severities (critical, major, minor), corrective "
            "actions, and deadlines. Flag any critical violations prominently."
        ),
        "examples": [
            {
                "question": "What happened during the Chicago food safety inspection? Were there any critical violations?",
                "guideline": "Must reference a specific inspection report by date or ID. Must state the inspection score and grade. Must explicitly state whether critical violations were found and cite at least one violation code if they exist.",
            },
            {
                "question": "Which locations failed inspection in the last quarter?",
                "guideline": "Must list specific locations with grades below B or scores below the passing threshold. Must include the inspection date and overall score for each location named.",
            },
            {
                "question": "What are the most common food safety violations across our network?",
                "guideline": "Must aggregate violations by category (food handling, sanitation, equipment, temperature, etc.). Must cite the most frequent violation codes and how often they appeared.",
            },
        ],
    },
    {
        "display_name": f"{CATALOG}-menu-knowledge",
        "key": "menu",
        "volume_path": f"/Volumes/{CATALOG}/menu_documents/menus",
        "source_display_name": "Menu PDFs",
        "description": (
            "Answers questions about Casper's Kitchens restaurant menus including dishes, "
            "ingredients, nutrition, allergens, and preparation details. Covers all 16 "
            "restaurant brands."
        ),
        "source_description": (
            "Restaurant menu PDFs for 16 ghost kitchen brands. Each PDF contains the full menu "
            "with item names, descriptions, prices, nutritional information (calories, protein, "
            "fat, carbs), and allergen warnings."
        ),
        "instructions": (
            "You are a restaurant menu assistant for Casper's Kitchens. "
            "Always cite which brand the information comes from. "
            "Be specific about allergen warnings and nutritional details."
        ),
        "examples": [
            {
                "question": "Which menu items are gluten-free and under $15?",
                "guideline": "Must list specific items with brand, dish name, price, and allergen status. Must filter to items below the price threshold.",
            },
            {
                "question": "Compare protein content across all burgers in our network",
                "guideline": "Must return per-item protein values with brand and dish name across multiple brands. Sort or rank by protein content.",
            },
            {
                "question": "Which dishes contain peanuts? List by brand.",
                "guideline": "Must list specific items with brand and dish name. Must cite allergen warnings explicitly. Group results by brand.",
            },
        ],
    },
    {
        "display_name": f"{CATALOG}-legal",
        "key": "legal",
        "volume_path": f"/Volumes/{CATALOG}/legal_complaints/documents",
        "source_display_name": "Legal Complaint Documents",
        "description": (
            "Answers questions about Casper's Kitchens legal exposure. Covers employment "
            "disputes (discrimination, wrongful termination, wage violations), customer "
            "liability claims, and vendor/contract disputes across all 8 locations (US + EMEA)."
        ),
        "source_description": (
            "Legal complaint PDFs for Casper's Kitchens Inc. Each document is a confidential "
            "attorney-client privileged case file covering case number, parties, legal basis, "
            "applicable statutes, relief sought, case status, and risk assessment."
        ),
        "instructions": (
            "You are a legal intelligence assistant for Casper's Kitchens. "
            "Always cite the specific case number and location when answering. "
            "Clearly state the risk level (HIGH/MEDIUM/LOW) and amount at stake. "
            "Remind users that all legal decisions should be made with counsel. "
            "Never speculate on legal outcomes."
        ),
        "examples": [
            {
                "question": "What active high-risk legal cases do we have? What is the total exposure?",
                "guideline": "Must list cases classified as HIGH risk. Must cite case numbers (CK-XX-XXXX), parties, and amount at stake. Must include the disclaimer about consulting counsel.",
            },
            {
                "question": "Are there any wrongful termination cases pending?",
                "guideline": "Must filter to cases citing wrongful termination. Must include case number, location, status, and risk level. Must cite the legal basis from the case file.",
            },
            {
                "question": "Which location has the most active legal complaints?",
                "guideline": "Must aggregate active cases by location. Must rank locations by case count. Must cite at least one specific case number per location named.",
            },
        ],
    },
    {
        "display_name": f"{CATALOG}-regulatory",
        "key": "regulatory",
        "volume_path": f"/Volumes/{CATALOG}/regulatory/documents",
        "source_display_name": "Regulatory Documents",
        "description": (
            "Answers questions about Casper's Kitchens regulatory compliance status. "
            "Covers food service permits, fire safety certificates, zoning compliance, "
            "FDA registrations, and food handler certifications across all 8 locations (US + EMEA)."
        ),
        "source_description": (
            "Regulatory compliance PDFs for all Casper's Kitchens ghost kitchen locations. "
            "Includes document type, issuing authority, issue date, expiry date, current status, "
            "and specific conditions and requirements from each regulatory body."
        ),
        "instructions": (
            "You are a regulatory compliance assistant. "
            "Always cite the specific document ID, issuing authority, and expiry date. "
            "Flag any permits or certificates that are expiring within 60 days or have a conditional status. "
            "Summarize compliance risk clearly and concisely."
        ),
        "examples": [
            {
                "question": "Which permits or certificates are expiring in the next 60 days?",
                "guideline": "Must list specific document IDs, issuing authorities, expiry dates, and locations. Sort by expiry date (soonest first).",
            },
            {
                "question": "Which locations are operating under a conditional permit?",
                "guideline": "Must list locations with conditional or restricted permits. Must cite the document ID, issuing authority, and the specific conditions imposed.",
            },
            {
                "question": "What's our FDA registration status across US locations?",
                "guideline": "Must cite document IDs and current status for each US location. Must flag any expired or conditional registrations.",
            },
        ],
    },
    {
        "display_name": f"{CATALOG}-audits",
        "key": "audits",
        "volume_path": f"/Volumes/{CATALOG}/audits/reports",
        "source_display_name": "Audit Reports",
        "description": (
            "Answers questions about Casper's Kitchens audit findings. "
            "Covers financial statement audits, operational compliance, food safety management, "
            "and supply chain audits across all locations and quarters."
        ),
        "source_description": (
            "Independent audit reports prepared by Big 4 and mid-tier firms for Casper's Kitchens. "
            "Each report includes audit type, period, scope, auditor's opinion, individual findings "
            "(Critical/Significant/Minor/Informational) with remediation details."
        ),
        "instructions": (
            "You are an audit intelligence assistant. "
            "Always cite the specific audit ID, auditing firm, and period covered. "
            "Highlight Critical and Significant findings prominently. "
            "Summarize the auditor's opinion and any patterns across multiple audits. "
            "Be specific about financial impacts and remediation deadlines."
        ),
        "examples": [
            {
                "question": "What were the most significant audit findings this quarter?",
                "guideline": "Must cite audit ID, auditing firm, and audit period. Must filter to Critical and Significant findings. Must summarize the auditor's opinion and remediation status.",
            },
            {
                "question": "Which audits received a qualified opinion?",
                "guideline": "Must list audits where the auditor issued a qualified or adverse opinion. Must cite firm, period, and the basis for qualification.",
            },
            {
                "question": "What's our remediation status on Critical findings from the past year?",
                "guideline": "Must filter to Critical findings opened in the last 12 months. Must group by remediation status (open / in-progress / closed). Must cite specific audit IDs.",
            },
        ],
    },
    {
        "display_name": f"{CATALOG}-consultancy",
        "key": "consultancy",
        "volume_path": f"/Volumes/{CATALOG}/consultancy/reports",
        "source_display_name": "Consultancy Reports",
        "description": (
            "Answers questions about strategic recommendations from management consultants. "
            "Covers market expansion, operations efficiency, AI transformation, and workforce "
            "management reports from top-tier consulting firms."
        ),
        "source_description": (
            "Management consulting reports prepared exclusively for Casper's Kitchens. "
            "Each report includes executive summary, detailed section-by-section analysis, "
            "financial projections, and specific recommendations with ROI estimates."
        ),
        "instructions": (
            "You are a strategic intelligence assistant. "
            "Always cite the consulting firm and report date. "
            "Summarize key recommendations with financial impact figures. "
            "When multiple reports address the same topic, synthesize the consensus view. "
            "Be direct and executive-ready in your summaries."
        ),
        "examples": [
            {
                "question": "What do consultants recommend as our top AI investments for the next 90 days?",
                "guideline": "Must cite the consulting firm and report date. Must include at least one concrete recommendation with a financial metric. Must frame in the 90-day horizon.",
            },
            {
                "question": "Which markets do consultants recommend for expansion?",
                "guideline": "Must cite the consulting firm and report. Must list specific markets with the rationale and investment estimate per market.",
            },
            {
                "question": "What's the consensus view on automation across our consulting reports?",
                "guideline": "Must synthesize across all consultancy reports addressing automation. Must cite specific reports. Must highlight where firms agree vs. disagree.",
            },
        ],
    },
]

KA_CONFIGS = [cfg for cfg in KA_CONFIGS_ALL if cfg.get("key") in KA_KEY_LIST]
if not KA_CONFIGS:
    raise ValueError(f"No KA configs selected from KA_KEYS={KA_KEY_LIST!r}")
print(f"Deploying {len(KA_CONFIGS)} Knowledge Assistant(s): {[c['key'] for c in KA_CONFIGS]}")


# KA instructions are the source of truth in KA_CONFIGS above — not
# round-tripped through the MLflow Prompt Registry.  Earlier versions of
# this stage also registered each KA's `instructions` under
# `{CATALOG}.prompts.ka_<short>_instructions` so domain experts could edit
# them in Catalog Explorer; that path was dropped because (a) it created
# six extra prompts that nobody iterates on in practice, and (b) the
# load_prompt → cfg["instructions"] override + PATCH-on-drift roundtrip
# was a lot of moving parts for zero observed benefit in demos.  Refund,
# Complaint, and Supervisor still register their prompts — KAs do not.


##### Create the KAs in parallel

Looks up existing KAs in `uc_state` first to skip already-created ones, then
queries the KA API once for anything still missing. New KAs are created
concurrently. The downstream `Readiness_Check` task polls the endpoints.

In [ ]:
# Resolve already-existing KAs by display_name.
# Source 1: uc_state (cheap, scoped to this catalog).
# Source 2: workspace-wide List Knowledge Assistants API (v2.1 has this; v2.0 didn't).
# Each entry maps display_name -> {"id", "endpoint_name"}.
known_kas = {}

try:
    df = spark.sql(f"""
        SELECT resource_data FROM {CATALOG}._internal_state.resources
        WHERE resource_type = 'knowledge_assistants'
        ORDER BY created_at DESC
    """)
    for row in df.collect():
        info = json.loads(row.resource_data)
        dn = info.get("name", "")  # uc_state stores display_name under "name" for back-compat
        kid = info.get("tile_id", "")
        ep = info.get("endpoint_name", "")
        if dn and kid and dn not in known_kas:
            known_kas[dn] = {"id": kid, "endpoint_name": ep}
    print(f"uc_state: found {len(known_kas)} KAs")
except Exception as e:
    print(f"\u26a0\ufe0f uc_state lookup failed: {e}")

target_names = {cfg["display_name"] for cfg in KA_CONFIGS}
missing_names = target_names - set(known_kas)
if missing_names:
    print(f"{len(missing_names)} KAs missing from uc_state \u2014 listing workspace KAs via {API_BASE}\u2026")
    try:
        params = {}
        while True:
            resp = w.api_client.do("GET", API_BASE, query=params)
            for ka in resp.get("knowledge_assistants", []):
                dn = ka.get("display_name", "")
                if dn in target_names and dn not in known_kas:
                    known_kas[dn] = {
                        "id": ka.get("id", ""),
                        "endpoint_name": ka.get("endpoint_name", ""),
                    }
            token = resp.get("next_page_token")
            if not token:
                break
            params = {"page_token": token}
    except Exception as e:
        print(f"\u26a0\ufe0f KA list error: {e}")


def _create_one(cfg):
    display_name = cfg["display_name"]

    # If a KA with this display_name already exists, GET its current
    # `instructions` and compare to what cfg["instructions"] resolves to
    # (load_prompt may have fetched a newer registered version above).
    # On drift, PATCH the KA via the v2.1 update endpoint so the new
    # prompt takes effect on the next request — no delete + recreate.
    if display_name in known_kas:
        existing = known_kas[display_name]
        try:
            ka_get = w.api_client.do("GET", f"{API_BASE}/{existing['id']}") or {}
            current_instructions = (ka_get.get("instructions") or "").strip()
            target_instructions = (cfg["instructions"] or "").strip()
            if current_instructions and current_instructions != target_instructions:
                print(f"[{display_name}] \u26a0\ufe0f  Instructions drift \u2014 PATCHing KA\u2026")
                try:
                    w.api_client.do(
                        "PATCH",
                        f"{API_BASE}/{existing['id']}",
                        query={"update_mask": "instructions"},
                        body={"instructions": cfg["instructions"]},
                    )
                    print(f"[{display_name}] \u2705 PATCH succeeded \u2014 KA now uses the registered prompt")
                except Exception as _patch_exc:
                    print(
                        f"[{display_name}] \u26a0\ufe0f  PATCH failed ({type(_patch_exc).__name__}: {_patch_exc}). "
                        f"As a fallback, delete this KA from the workspace and redeploy this stage."
                    )
            else:
                print(f"[{display_name}] \u267b\ufe0f  Already exists (id={existing['id']}, instructions match) \u2014 skipping update")
        except Exception as _drift_exc:
            print(
                f"[{display_name}] \u267b\ufe0f  Already exists (id={existing['id']}) \u2014 skipping update "
                f"(drift check failed: {type(_drift_exc).__name__}: {_drift_exc})"
            )
        return {
            "display_name": display_name,
            "id": existing["id"],
            "endpoint_name": existing["endpoint_name"],
        }

    # Step 1: create the Knowledge Assistant. v2.1 body is just metadata; the
    # endpoint_name is assigned by the API and returned in the response.
    print(f"[{display_name}] Creating KA\u2026")
    ka_body = {
        "display_name": display_name,
        "description": cfg["description"],
        "instructions": cfg["instructions"],
    }
    ka_resp = w.api_client.do("POST", API_BASE, body=ka_body) or {}
    ka_id = ka_resp.get("id", "")
    ka_endpoint = ka_resp.get("endpoint_name", "")
    if not ka_id:
        raise RuntimeError(f"[{display_name}] Create response missing id: {ka_resp}")

    # If the endpoint name wasn't populated synchronously, re-fetch.
    if not ka_endpoint:
        try:
            ka_get = w.api_client.do("GET", f"{API_BASE}/{ka_id}") or {}
            ka_endpoint = ka_get.get("endpoint_name", "")
        except Exception:
            pass

    # Step 2: attach a "files" knowledge source pointing at the UC Volume.
    # The Readiness_Check task is responsible for waiting until the source
    # finishes its initial sync.
    src_body = {
        "display_name": cfg["source_display_name"],
        "description": cfg["source_description"],
        "source_type": "files",
        "files": {"path": cfg["volume_path"]},
    }
    try:
        w.api_client.do("POST", f"{API_BASE}/{ka_id}/knowledge-sources", body=src_body)
        print(f"[{display_name}] \u2705 Created (id={ka_id}, endpoint={ka_endpoint or 'pending'})")
    except Exception as e:
        print(f"[{display_name}] \u26a0\ufe0f  KA created but source attach failed: {e}")

    return {"display_name": display_name, "id": ka_id, "endpoint_name": ka_endpoint}


# Serial creation, NOT parallel.  Parallel `_create_one` invocations
# (previously via ThreadPoolExecutor(max_workers=6)) collided in the
# Databricks workspace tree because the KA Manager API does multiple
# /workspaces:<id> mutations per KA and isn't reliably safe under
# concurrent writes — surfaces as:
#   InternalError: [OCC check failure] A descendant or edge of
#   /workspaces:<id> was inserted or deleted during the transaction.
# Serialising adds a few seconds per KA (so ~30s-1min for 6 KAs) and
# eliminates the race entirely.
created_agents = [_create_one(cfg) for cfg in KA_CONFIGS]

# uc_state stores: name=display_name (for routing lookups), tile_id=KA UUID
# (for delete + readiness checks), endpoint_name=auto-generated by API.
#
# `uc_state.add()` is a raw INSERT (no upsert), so re-running this stage
# would append duplicate rows for KAs we've already registered.  Look up
# existing tile_ids first and skip the add() call when the KA is already
# tracked — keeps uc_state at exactly one row per KA across re-runs.
existing_ka_tile_ids = set()
try:
    _existing_rows = spark.sql(f"""
        SELECT resource_data:tile_id::string AS tile_id
        FROM {CATALOG}._internal_state.resources
        WHERE resource_type = 'knowledge_assistants'
    """).collect()
    existing_ka_tile_ids = {row.tile_id for row in _existing_rows if row.tile_id}
except Exception as _enum_exc:
    print(f"⚠️  Could not enumerate existing uc_state KA rows ({type(_enum_exc).__name__}: {_enum_exc}) — falling back to unconditional add() (may produce duplicates).")

for r in created_agents:
    if r["id"] in existing_ka_tile_ids:
        print(f"   ♻  Already in uc_state: {r['display_name']} (tile_id={r['id']}) — skipping add()")
        continue
    add(CATALOG, "knowledge_assistants", {
        "endpoint_name": r["endpoint_name"],
        "tile_id": r["id"],
        "name": r["display_name"],
    })
    print(f"   Registered: {r['display_name']} (id={r['id']}, endpoint={r['endpoint_name'] or 'pending'})")

print(f"\n\u2705 Knowledge Agents stage complete \u2014 {len(created_agents)} KAs processed")

##### Attach example questions to each KA

Idempotent: lists existing examples, deletes them, then re-POSTs the desired set
defined in `KA_CONFIGS[i]["examples"]`. This way a manual edit (or wipe) heals
on the next deploy and we don't accumulate duplicates.

**API**: `POST /api/2.0/knowledge-assistants/{tile_id}/examples` — body
`{"tile_id": ka_id, "question": str, "guidelines": [str]}` (one example per
request). Same shape as the supervisor examples API, just on the KA surface
(note: examples live on `v2.0` even though the rest of this stage uses `v2.1`).

In [ ]:
import time as _time

_KA_EXAMPLES_BASE_TPL = "/api/2.0/knowledge-assistants/{ka_id}/examples"


def _format_api_error(e: Exception) -> str:
    """str(e) on databricks-sdk errors is sometimes empty/None when the server
    returns a non-standard error body. Surface enough detail to debug."""
    msg = str(e) or ""
    if not msg or msg == "None":
        msg = repr(e)
    code = getattr(e, "error_code", None)
    return f"{type(e).__name__}: {msg}" + (f" [error_code={code}]" if code else "")


def _do_with_retry(method: str, path: str, body: dict | None = None, retries: int = 1, backoff_s: float = 2.0):
    """api_client.do with one retry on Exception. Used to absorb transient
    rate-limit failures on the KA examples API which empirically rejects
    bursts of >~5 req/s with empty error bodies."""
    last_exc = None
    for attempt in range(retries + 1):
        try:
            if body is None:
                return w.api_client.do(method, path)
            return w.api_client.do(method, path, body=body)
        except Exception as e:
            last_exc = e
            if attempt < retries:
                _time.sleep(backoff_s)
    raise last_exc


cfg_by_name = {cfg["display_name"]: cfg for cfg in KA_CONFIGS}

for r in created_agents:
    display_name = r["display_name"]
    ka_id = r["id"]
    cfg = cfg_by_name.get(display_name)
    if not cfg or not ka_id:
        continue
    examples = cfg.get("examples") or []
    if not examples:
        continue

    base = _KA_EXAMPLES_BASE_TPL.format(ka_id=ka_id)

    existing = []
    try:
        resp = w.api_client.do("GET", base) or {}
        existing = resp.get("examples", []) or []
    except Exception as _le:
        print(f"[{display_name}] (skipping pre-clear: {_format_api_error(_le)})")
    deleted, delete_failures = 0, 0
    for ex in existing:
        ex_id = ex.get("example_id") or ex.get("id")
        if not ex_id:
            continue
        try:
            _do_with_retry("DELETE", f"{base}/{ex_id}", retries=1, backoff_s=2.0)
            deleted += 1
        except Exception as _de:
            delete_failures += 1
            print(f"[{display_name}] \u26a0\ufe0f  Could not delete existing example {ex_id}: {_format_api_error(_de)}")
        _time.sleep(0.4)  # gentle pacing — examples API rate-limits bursts
    if existing:
        print(f"[{display_name}] Cleared {deleted}/{len(existing)} existing example(s) before re-attach"
              f"{f' (delete failures: {delete_failures})' if delete_failures else ''}")

    posted = 0
    failures = 0
    for ex in examples:
        guideline = ex.get("guideline")
        guidelines = ex.get("guidelines")
        if guidelines is None and guideline is not None:
            guidelines = [guideline]
        body = {"tile_id": ka_id, "question": ex["question"]}
        if guidelines:
            body["guidelines"] = guidelines
        try:
            _do_with_retry("POST", base, body=body, retries=1, backoff_s=2.0)
            posted += 1
        except Exception as _pe:
            failures += 1
            print(f"[{display_name}] \u26a0\ufe0f  Failed to attach {ex['question']!r}: {_format_api_error(_pe)}")
        _time.sleep(0.4)  # gentle pacing — examples API rate-limits bursts

    if failures:
        print(f"[{display_name}] \u26a0\ufe0f  Examples attached partially: {posted}/{len(examples)} (failures={failures})")
    else:
        print(f"[{display_name}] \u2705 Examples attached ({posted}/{len(examples)})")